# Activité Pratique N°4 — Architecture Transformer
## Partie 1 — Étape 6 : Modèle de type BERT

**IA Générative Multi Modale — 5IAD G1, EMSI** · Étudiant : BENHAYOUN SADAFI Mohamed

Créer et tester un exemple basé sur un Transformer de type BERT : prédire un mot caché dans une séquence.

BERT est un **encodeur bidirectionnel** : pour remplacer `[MASK]`, il regarde le contexte **à gauche et à droite**.
On utilise `bert-base-multilingual-cased`, qui comprend le français.

```
Input : « Rabat est [MASK] du Maroc » ──► Bidirectional Encoder Representations from Transformers ──► Output : « capitale »
```

In [1]:
import torch
from transformers import AutoModelForMaskedLM, AutoTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")
modele = AutoModelForMaskedLM.from_pretrained("bert-base-multilingual-cased").to(device).eval()
print("Token de masque :", tokenizer.mask_token)

C:\Users\user\Desktop\5IAD G1 BENHAYOUN SADAFI Mohamed, emsi, matiere ia multimodale\Activité Pratique N°4 - Architecture Transformer\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Users\user\Desktop\5IAD G1 BENHAYOUN SADAFI Mohamed, emsi, matiere ia multimodale\Activité Pratique N°4 - Architecture Transformer\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--bert-base-multilingual-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


C:\Users\user\Desktop\5IAD G1 BENHAYOUN SADAFI Mohamed, emsi, matiere ia multimodale\Activité Pratique N°4 - Architecture Transformer\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Loading weights:   3%|▎         | 6/202 [00:00<00:04, 39.29it/s]

Loading weights:   5%|▍         | 10/202 [00:00<00:05, 38.39it/s]

... (4 outputs left out)

Loading weights:  12%|█▏        | 25/202 [00:01<00:12, 14.42it/s]

Loading weights: 100%|██████████| 202/202 [00:01<00:00, 142.06it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-multilingual-cased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Token de masque : [MASK]


In [2]:
def completer(phrase, k=5):
    phrase = phrase.replace("[MASK]", tokenizer.mask_token)
    entrees = tokenizer(phrase, return_tensors="pt").to(device)
    print(phrase)
    print("  tokens :", tokenizer.convert_ids_to_tokens(entrees["input_ids"][0]))
    position = (entrees["input_ids"][0] == tokenizer.mask_token_id).nonzero(as_tuple=True)[0].item()
    with torch.no_grad():
        logits = modele(**entrees).logits[0, position]       # scores sur tout le vocabulaire
    top = torch.topk(torch.softmax(logits, dim=-1), k)
    for p, i in zip(top.values, top.indices):
        mot = tokenizer.decode([i.item()]).strip()
        print(f"  {p.item():6.2%}  {phrase.replace(tokenizer.mask_token, mot)}")
    print()


completer("Rabat est [MASK] du Maroc.")
completer("Rabat est la [MASK] du Maroc.")

Rabat est [MASK] du Maroc.
  tokens : ['[CLS]', 'Rabat', 'est', '[MASK]', 'du', 'Maroc', '.', '[SEP]']


  31.72%  Rabat est capitale du Maroc.
  19.48%  Rabat est Rabat du Maroc.
  12.39%  Rabat est ville du Maroc.
   5.37%  Rabat est préfecture du Maroc.
   4.71%  Rabat est Casablanca du Maroc.

Rabat est la [MASK] du Maroc.
  tokens : ['[CLS]', 'Rabat', 'est', 'la', '[MASK]', 'du', 'Maroc', '.', '[SEP]']
  86.06%  Rabat est la capitale du Maroc.
   5.19%  Rabat est la ville du Maroc.
   5.03%  Rabat est la préfecture du Maroc.
   0.85%  Rabat est la Rabat du Maroc.
   0.58%  Rabat est la capital du Maroc.



### Le contexte de droite compte

In [3]:
completer("Le [MASK] mange du fromage.")
completer("La [MASK] est connectée à l'ordinateur.")
completer("Casablanca est la plus grande [MASK] du Maroc.")

Le [MASK] mange du fromage.


  tokens : ['[CLS]', 'Le', '[MASK]', 'mange', 'du', 'from', '##age', '.', '[SEP]']
  19.90%  Le village mange du fromage.
   3.45%  Le hameau mange du fromage.
   3.45%  Le domaine mange du fromage.
   2.68%  Le bourg mange du fromage.
   2.57%  Le principal mange du fromage.

La [MASK] est connectée à l'ordinateur.
  tokens : ['[CLS]', 'La', '[MASK]', 'est', 'connect', '##ée', 'à', 'l', "'", 'ord', '##inateur', '.', '[SEP]']


   9.91%  La machine est connectée à l'ordinateur.
   2.85%  La transmission est connectée à l'ordinateur.
   2.76%  La ligne est connectée à l'ordinateur.
   2.55%  La batterie est connectée à l'ordinateur.
   2.16%  La carte est connectée à l'ordinateur.

Casablanca est la plus grande [MASK] du Maroc.
  tokens : ['[CLS]', 'Casablanca', 'est', 'la', 'plus', 'grande', '[MASK]', 'du', 'Maroc', '.', '[SEP]']
  61.64%  Casablanca est la plus grande ville du Maroc.
  29.21%  Casablanca est la plus grande Casablanca du Maroc.
   1.88%  Casablanca est la plus grande port du Maroc.
   1.44%  Casablanca est la plus grande capitale du Maroc.
   0.62%  Casablanca est la plus grande agglomération du Maroc.



| | GPT | BERT |
|---|---|---|
| Architecture | Décodeur seul | Encodeur seul |
| Attention | Causale | Bidirectionnelle |
| Apprentissage | Prédire le token suivant | Prédire les tokens masqués |
| Usage | Génération, dialogue, code | Classification, extraction, compréhension |